# 32 · Feature engineering sobre E3·S4_s2: inicios de brote

Encargo `docs/modeling/fe_hiperparametros.md`. La arquitectura (E3·S4_s2) y el protocolo
(`protocolo_arquitectura.py`) quedan fijos; este notebook solo orquesta. Nada se escribe en `data/`:
las variables nuevas se calculan en memoria.

## Fase 0 · Preparación y línea base (referencia B)

1. Reproducir E3·S4_s2 con `base_6` y las 5 semillas, **reentrenando** (sin caché del notebook 31), y
   comparar con `fase5_decision.metricas_mediana_semillas` al redondeo de 3 decimales.
2. Guardar la referencia B: medianas y rangos de semillas, antelación y recall en brotes nuevos.
3. Inventario de las candidatas de gold h=2 y sus antecedentes con S1 (ablaciones 25–29).
4. Tiempo de un ajuste completo (5 bloques × 1 semilla).

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.eda.carga import cargar_integrado
from src.modeling import cache_experimentos as cache
from src.modeling import diagnostico_arquitectura as d
from src.modeling import estrategias_horizonte as eh
from src.modeling import fe_hiperparametros as fh
from src.modeling import protocolo_arquitectura as pa
from src.modeling import salida_probabilistica as sp
from src.modeling.dispositivo import configuracion_xgboost
from src.modeling.seguimiento_mlflow import registrar_o_avisar
from src.utils.paths import (DOCS, EXPERIMENTOS_FE_HIPERPARAMETROS, GOLD, GOLD_MANIFEST, METRICAS_ARQUITECTURA,
                             METRICAS_FE_HIPERPARAMETROS, SILVER_INTEGRADO)

warnings.filterwarnings("ignore", message=".*mismatched devices.*")
pd.set_option("display.precision", 3)
HUELLA = cache.huella_encargo()
CACHE = EXPERIMENTOS_FE_HIPERPARAMETROS / "cache"
DESTINO0 = EXPERIMENTOS_FE_HIPERPARAMETROS / "fase0"
print("dispositivo:", configuracion_xgboost(), "| huella:", HUELLA[:12])

panel = cargar_integrado(SILVER_INTEGRADO)
regla = json.loads(GOLD_MANIFEST.read_text(encoding="utf-8"))["regla_brote"]
ins = fh.insumos(panel, regla)
matrices, serie = ins["matrices"], ins["serie"]
pd.DataFrame([{"horizonte": h, "filas": len(m), "causal": bool((m.semana_inicio > m.origen_cierre).all())}
              for h, m in matrices.items()])

dispositivo: {'device': 'cuda', 'n_jobs': 12} | huella: 0fb86c676a18


,horizonte,filas,causal
0,2,30160,True
1,3,30095,True
2,4,30030,True


### 0.3 · Reproducción sin caché

`forzar=True`: siempre reentrena las 5 semillas, aunque exista una caché con la misma huella
(el resultado queda guardado solo para que las fases siguientes lo reutilicen).

In [2]:
referencia = cache.cargar_o_calcular(
    "fase0_referencia_B", lambda: fh.evaluar_variante(matrices, serie), huella=HUELLA, directorio=CACHE, forzar=True)
resumen_b = fh.resumir_variante(referencia["evaluaciones"], serie)
guardada = json.loads(METRICAS_ARQUITECTURA.read_text(encoding="utf-8"))["fase5_decision"]["metricas_mediana_semillas"]
reproduccion = fh.comparar_reproduccion(resumen_b["medianas"], guardada)
print(f"métricas comparadas: {len(reproduccion)} | coinciden a 3 decimales: {int(reproduccion.coincide.sum())}"
      f" | diferencia absoluta máxima: {reproduccion.diferencia_absoluta.max():.2e}")
reproduccion[~reproduccion.coincide]

caché: 'fase0_referencia_B' recalculada (forzada) (huella 0fb86c676a18)


métricas comparadas: 615 | coinciden a 3 decimales: 615 | diferencia absoluta máxima: 4.99e-11


,horizonte,bloque,metrica,guardada,reproducida,diferencia_absoluta,coincide


In [3]:
# Si algo no coincide, el encargo se detiene aquí y se reporta (criterio acordado con Rosa).
if not reproduccion.coincide.all():
    raise RuntimeError("E3·S4_s2 no reproduce fase5_decision al redondeo de 3 decimales")

### 0.4 · Referencia B

Medianas de semillas por horizonte y temporada, con el rango entre semillas (umbral de empate de la regla
simétrica). 2021 es calibración, 2022–2023 selección, 2024 contraste y 2025 sensibilidad de conteos.

In [4]:
COLUMNAS_B = ["horizonte", "bloque", "precision_050", "recall_050", "f1_050", "auprc", "brier",
              "razon_mae", "razon_rmse", "f1_persistencia", "auprc_persistencia"]
medianas_b, rangos_b = resumen_b["medianas"], resumen_b["rangos"]
medianas_b[COLUMNAS_B]

,horizonte,bloque,precision_050,recall_050,f1_050,auprc,brier,razon_mae,razon_rmse,f1_persistencia,auprc_persistencia
0,2,calendario_2025,0.000,0.000,0.000,0.082,0.003,2.016,1.519,0.000,0.010
1,2,temporada_2021,NaN,NaN,NaN,NaN,NaN,1.188,1.054,0.610,0.623
2,2,temporada_2022,0.794,0.601,0.682,0.796,0.051,1.056,1.173,0.686,0.676
3,2,temporada_2023,0.919,0.782,0.845,0.948,0.063,1.005,1.174,0.861,0.899
4,2,temporada_2024,0.750,0.894,0.815,0.894,0.073,1.349,1.780,0.798,0.843
5,3,calendario_2025,0.100,0.333,0.154,0.091,0.004,2.451,1.956,0.154,0.089
6,3,temporada_2021,NaN,NaN,NaN,NaN,NaN,1.198,1.085,0.514,0.502
7,3,temporada_2022,0.869,0.445,0.590,0.784,0.057,1.153,1.377,0.679,0.640
8,3,temporada_2023,0.936,0.716,0.812,0.929,0.077,0.865,0.963,0.829,0.868
9,3,temporada_2024,0.716,0.880,0.789,0.867,0.086,1.465,1.883,0.776,0.803


In [5]:
rangos_b

,horizonte,bloque,f1_050,auprc,brier,razon_mae,rmse,mae,recall_050,precision_050
0,2,calendario_2025,0.000,5.877e-03,1.127e-04,0.014,0.026,0.004,0.000,0.000
1,2,temporada_2021,NaN,NaN,NaN,0.023,0.127,0.012,NaN,NaN
2,2,temporada_2022,0.019,2.894e-03,5.189e-04,0.028,0.476,0.056,0.030,0.013
3,2,temporada_2023,0.004,7.404e-04,3.229e-04,0.028,1.728,0.299,0.006,0.003
4,2,temporada_2024,0.005,9.654e-04,5.265e-04,0.075,2.813,0.370,0.011,0.004
5,3,calendario_2025,0.167,1.474e-02,1.439e-04,0.026,0.047,0.008,0.333,0.111
6,3,temporada_2021,NaN,NaN,NaN,0.011,0.100,0.007,NaN,NaN
7,3,temporada_2022,0.008,3.053e-03,6.976e-04,0.037,0.549,0.089,0.014,0.021
8,3,temporada_2023,0.005,7.533e-04,3.664e-04,0.013,0.600,0.192,0.009,0.004
9,3,temporada_2024,0.004,2.857e-03,9.174e-04,0.049,2.476,0.314,0.015,0.004


Antelación en inicios limpios (fracción de episodios avisados con ≥ 2 semanas): mediana y rango de
semillas, junto a la persistencia. Debajo, la diferencia pareada modelo − persistencia con IC bootstrap
(semilla 17, como en el notebook 31).

In [6]:
resumen_b["antelacion"]

,horizonte,bloque,mediana,minimo,maximo,episodios,persistencia
0,2,seleccion_2022_2023,0.084,0.084,0.095,95,0.116
1,2,temporada_2022,0.053,0.053,0.079,38,0.105
2,2,temporada_2023,0.105,0.105,0.105,57,0.123
3,2,temporada_2024,0.174,0.174,0.174,23,0.130
4,3,seleccion_2022_2023,0.095,0.084,0.105,95,0.137
5,3,temporada_2022,0.053,0.026,0.053,38,0.105
6,3,temporada_2023,0.140,0.123,0.140,57,0.158
7,3,temporada_2024,0.217,0.174,0.304,23,0.087
8,4,seleccion_2022_2023,0.105,0.105,0.116,95,0.211
9,4,temporada_2022,0.000,0.000,0.000,38,0.263


In [7]:
antelacion_pers_b = fh.diferencia_antelacion_persistencia(referencia["evaluaciones"][17])
antelacion_pers_b

,horizonte,episodios,diferencia,ic95_inferior,ic95_superior,diferencia_temporada_2022,diferencia_temporada_2023
0,2,95,-0.032,-0.084,0.021,-0.053,-0.018
1,3,95,-0.042,-0.105,0.011,-0.053,-0.035
2,4,95,-0.105,-0.189,-0.021,-0.263,0.000


Recall en las 2 primeras semanas de brotes nuevos (inicio limpio) frente a semanas de brotes en curso
(alerta p ≥ 0,50 calibrada). Se reporta siempre; no decide.

In [8]:
recall_b = resumen_b["recall_por_tipo"]
recall_b.pivot_table(index=["horizonte", "tipo"], columns="bloque", values="recall_mediana").round(3)

bloque                 calendario_2025  temporada_2022  temporada_2023  \
horizonte tipo                                                           
2         brote_nuevo              0.0           0.061           0.093   
          en_curso                 0.0           0.670           0.844   
3         brote_nuevo              0.0           0.041           0.093   
          en_curso                 1.0           0.500           0.772   
4         brote_nuevo              0.0           0.000           0.081   
          en_curso                 0.0           0.369           0.717   

bloque                 temporada_2024  
horizonte tipo                         
2         brote_nuevo           0.206  
          en_curso              0.922  
3         brote_nuevo           0.147  
          en_curso              0.910  
4         brote_nuevo           0.118  
          en_curso              0.898

In [9]:
cumplimiento_b = sp.cumplimiento_hu0007_4(medianas_b[medianas_b.bloque.isin([*pa.BLOQUES_SELECCION, pa.BLOQUE_CONTRASTE])])
cumplimiento_b

,horizonte,bloque,precision_050,recall_050,f1_050,razon_mae,razon_rmse,cumple_precision_050,cumple_recall_050,cumple_f1_050,cumple_razon_mae,cumple_razon_rmse,cumple_clasificacion,cumple_regresion
2,2,temporada_2022,0.794,0.601,0.682,1.056,1.173,True,False,False,False,False,False,False
3,2,temporada_2023,0.919,0.782,0.845,1.005,1.174,True,False,True,False,False,False,False
4,2,temporada_2024,0.750,0.894,0.815,1.349,1.780,True,True,True,False,False,True,False
7,3,temporada_2022,0.869,0.445,0.590,1.153,1.377,True,False,False,False,False,False,False
8,3,temporada_2023,0.936,0.716,0.812,0.865,0.963,True,False,True,False,False,False,False
9,3,temporada_2024,0.716,0.880,0.789,1.465,1.883,True,True,True,False,False,True,False
12,4,temporada_2022,0.890,0.327,0.478,1.293,1.666,True,False,False,False,False,False,False
13,4,temporada_2023,0.941,0.664,0.779,0.793,0.846,True,False,True,True,True,False,True
14,4,temporada_2024,0.664,0.867,0.751,1.504,1.874,True,True,True,False,False,True,False


In [10]:
DESTINO0.mkdir(parents=True, exist_ok=True)
pd.concat(referencia["predicciones"].values(), ignore_index=True).to_csv(
    DESTINO0 / "predicciones_referencia_B.csv.gz", index=False)
sorted(p.name for p in DESTINO0.iterdir())

['predicciones_referencia_B.csv.gz']

### 0.5 · Candidatas de gold h=2 y antecedentes con S1

Familia y nulos de cada candidata predictora de gold h=2 (`contrato_xgboost.familias_predictoras`).
Las 6 de `base_6` están incluidas en el conteo.

In [11]:
gold_h2 = pd.read_csv(GOLD / f"{SILVER_INTEGRADO.stem}_h2_gold.csv", dtype={"ubigeo": "string"},
                      parse_dates=["semana_inicio"])
inventario = fh.inventario_candidatas(gold_h2, matrices[2])
display(inventario.groupby("familia").agg(columnas=("columna", "size"), nulos_gold_max=("nulos_gold", "max"),
                                          nulos_tras_arranque_max=("nulos_tras_arranque", "max")))
inventario

,columnas,nulos_gold_max,nulos_tras_arranque_max
familia,,,
calendario,2,0,0
clima,2,390,65
historia,4,325,0
jerarquia,4,130,0
socio_2017,4,0,0
socio_anual_reconstruida,16,0,0
vecinos,2,130,0


,columna,familia,en_base_6,nulos_gold,fraccion_nulos_gold,primera_semana_con_valor,nulos_tras_arranque
0,casos_lag_2,historia,True,130,0.004,2017-01-15,0
1,casos_lag_3,historia,True,195,0.006,2017-01-22,0
2,casos_media_4_h2,historia,True,325,0.011,2017-02-05,0
3,casos_semanas_positivas_4_h2,historia,True,325,0.011,2017-02-05,0
4,semana_epi_seno,calendario,True,0,0.000,2017-01-01,0
5,semana_epi_coseno,calendario,True,0,0.000,2017-01-01,0
6,vecinos_media_casos_knn5_h2,vecinos,False,130,0.004,2017-01-15,0
7,vecinos_frac_con_casos_knn5_h2,vecinos,False,130,0.004,2017-01-15,0
8,provincia_otros_casos_h2,jerarquia,False,130,0.004,2017-01-15,0
9,provincia_otros_frac_con_casos_h2,jerarquia,False,130,0.004,2017-01-15,0


**Antecedente, no comparable.** Las ablaciones 25–29 usaron S1 (regresor `log1p` + clasificador) y su propio
protocolo: umbral elegido en validación, F1 medio de 2021–2023 y una semilla, sin rangos de semillas ni regla
simétrica. Las diferencias de F1 frente a `base_6` se listan solo como contexto para la fase 2.

In [12]:
antecedentes = fh.antecedentes_s1(fh.cargar_ablaciones(DOCS / "modeling" / "metricas"))
antecedentes

,ablacion,variante,columnas_agregadas,elegida_en_validacion,f1_val_medio_reg,delta_f1_val_reg_vs_base_6,f1_val_medio_cls,delta_f1_val_cls_vs_base_6
0,25_espacial,base_6,[],NaN,0.677,0.000e+00,0.592,0.000e+00
1,25_espacial,mas_vecinos,"[vecinos_media_casos_knn5_h2, vecinos_frac_con...",NaN,0.680,3.215e-03,0.593,3.686e-04
2,25_espacial,mas_jerarquia,"[provincia_otros_casos_h2, provincia_otros_fra...","regresion, clasificacion",0.688,1.036e-02,0.604,1.184e-02
3,25_espacial,mas_ambos,"[vecinos_media_casos_knn5_h2, vecinos_frac_con...",NaN,0.667,-1.035e-02,0.595,2.362e-03
4,26_clima,base_6,[],"regresion, clasificacion",0.667,0.000e+00,0.592,0.000e+00
5,26_clima,mas_clima_observado,"[hum_rel_media_media5_h2, precip_total_mm_medi...",NaN,0.643,-2.371e-02,0.534,-5.727e-02
6,26_clima,mas_anomalias,"[hum_rel_media_anomalia_media5_h2, precip_tota...",NaN,0.637,-2.992e-02,0.494,-9.716e-02
7,26_clima,mas_ambos,"[hum_rel_media_media5_h2, precip_total_mm_medi...",NaN,0.627,-4.035e-02,0.518,-7.413e-02
8,27_sociodemografica,base_6,[],NaN,0.677,0.000e+00,0.592,0.000e+00
9,27_sociodemografica,fija_poblacion,[log_poblacion_censo_2017],"fija:regresion, fija:clasificacion",0.680,2.579e-03,0.655,6.277e-02


### 0.6 · Tiempo de un ajuste completo

5 bloques × 1 semilla (predicción E3·S4_s2) más la evaluación del protocolo, 3 repeticiones.

In [13]:
tiempos = fh.medir_ajuste(matrices, serie, repeticiones=3)
tiempos.describe().loc[["mean", "min", "max"]]

,repeticion,segundos_ajuste_prediccion,segundos_evaluacion,segundos_total
mean,2.0,4.417,2.556,6.974
min,1.0,4.251,2.440,6.775
max,3.0,4.545,2.704,7.161


### Registro (JSON y MLflow)

In [14]:
reporte_b = fh.reporte_mlflow("fe_hiperparametros_fase0_referencia_B", {fh.NOMBRE_REFERENCIA: medianas_b})
ruta = d.guardar_seccion_metricas(METRICAS_FE_HIPERPARAMETROS, "fase0_referencia", {
    "arquitectura": {"estrategia": fh.ESTRATEGIA, "formulacion": fh.FORMULACION, "columnas": list(eh.GENERICAS),
                     "hiperparametros": reporte_b["parametros"], "semillas": list(pa.SEMILLAS)},
    "reproduccion": {"criterio": "igualdad al redondeo de 3 decimales con fase5_decision.metricas_mediana_semillas",
                     "metricas_comparadas": int(len(reproduccion)), "coinciden": int(reproduccion.coincide.sum()),
                     "diferencia_absoluta_maxima": float(reproduccion.diferencia_absoluta.max()),
                     "reentrenada_sin_cache": True},
    "referencia_B": {"metricas_mediana_semillas": d.a_json(medianas_b), "rangos_semillas": d.a_json(rangos_b),
                     "antelacion": d.a_json(resumen_b["antelacion"]),
                     "antelacion_frente_a_persistencia_semilla_17": d.a_json(antelacion_pers_b),
                     "recall_por_tipo_semana": d.a_json(recall_b),
                     "cumplimiento_hu0007_4": d.a_json(cumplimiento_b)},
    "inventario_candidatas_gold_h2": d.a_json(inventario),
    "antecedentes_s1_ablaciones_25_29": {
        "nota": "S1 y protocolo propio (umbral elegido en validación, F1 medio 2021–2023, una semilla); "
                "antecedente, no comparable con los veredictos de este encargo",
        "variantes": d.a_json(antecedentes)},
    "tiempos_ajuste_completo_segundos": d.a_json(tiempos),
})
print(ruta.relative_to(ROOT))

docs\modeling\metricas\fe_hiperparametros.json


In [15]:
registrar_o_avisar("fe_hiperparametros_fase0_referencia_B", reporte_b, metricas_path=METRICAS_FE_HIPERPARAMETROS,
                   salida_dir=DESTINO0, gold_dir=None,
                   descripcion="Referencia B: E3·S4_s2 con base_6 e hiperparámetros fijos, mediana de 5 semillas")

MLflow: run b702b131199b480e9df8ce84f6978fa1 en 'denguekay/fe_hiperparametros_fase0_referencia_B' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'b702b131199b480e9df8ce84f6978fa1'